In [ ]:
import timm
import torch
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from huggingface_hub import login

In [ ]:
timm_kwargs = {
   'img_size': 224, 
   'patch_size': 14, 
   'depth': 24,
   'num_heads': 24,
   'init_values': 1e-5, 
   'embed_dim': 1536,
   'mlp_ratio': 2.66667*2,
   'num_classes': 0, 
   'no_embed_class': True,
   'mlp_layer': timm.layers.SwiGLUPacked, 
   'act_layer': torch.nn.SiLU, 
   'reg_tokens': 8, 
   'dynamic_img_size': True
}
model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)
transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
model.eval()

In [ ]:
import torch
import torchvision
import os
from os.path import join as j_
from PIL import Image
import pandas as pd
import numpy as np

# loading all packages here to start
from uni import get_encoder
from uni.downstream.extract_patch_features import extract_patch_features_from_dataloader
from uni.downstream.eval_patch_features.linear_probe import eval_linear_probe
from uni.downstream.eval_patch_features.fewshot import eval_knn, eval_fewshot
from uni.downstream.eval_patch_features.protonet import ProtoNet, prototype_topk_vote
from uni.downstream.eval_patch_features.metrics import get_eval_metrics, print_metrics
from uni.downstream.utils import concat_images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Creating tiles directly from image input without metadata file

In [ ]:
input_file = "data/image.tif"
tile_size = 128  # Example tile size

 # Load the image
Image.MAX_IMAGE_PIXELS = None  # Disable decompression bomb error for large images
image = Image.open(input_file)

# Get original dimensions
orig_width, orig_height = image.size
    # Calculate new dimensions divisible by tile_size
new_width = (orig_width // tile_size) * tile_size
new_height = (orig_height // tile_size) * tile_size

    # Calculate cropping box 
crop_left = (orig_width - new_width) // 2
crop_top = (orig_height - new_height) // 2
crop_right = orig_width - crop_left - new_width
crop_bottom = orig_height - crop_top - new_height

# Perform the crop
print("Performing cropping. Make take a while for large files")
crop_pic = image.crop((crop_left, crop_top, orig_width - crop_right, orig_height - crop_bottom))
crop_pic.size
crop_pic_array = np.array(crop_pic)
print(crop_pic_array.nbytes, "bytes")

In [ ]:
def create_image_tiles(input_file, output_folder, tile_size=256, file=crop_pic):

    Image.MAX_IMAGE_PIXELS = None
    image = file  # using the preloaded image
    img_width, img_height = image.size

    os.makedirs(output_folder, exist_ok=True)

    # Generate unique identifier based on the input file name 
    unique_id = os.path.splitext(os.path.basename(input_file))[0]

    total_tiles = (img_width // tile_size) * (img_height // tile_size)

    cpu_usage = []
    ram_usage = []

    # Get the current process for memory info
    process = psutil.Process()

    print(f"Generating tiles for image '{input_file}'...")
    start_time = time.time()  # Start timing

    with tqdm(total=total_tiles, desc="Processing Tiles", unit="tile") as pbar:
        tile_count = 0
        for y in range(0, img_height, tile_size):
            for x in range(0, img_width, tile_size):
                # Record current CPU and RAM usage before processing this tile
                cpu_percent = psutil.cpu_percent(interval=None)  # instantaneous snapshot (non-blocking)
                memory_mb = process.memory_info().rss / (1024 ** 2)  # Convert bytes to MB
                cpu_usage.append(cpu_percent)
                ram_usage.append(memory_mb)

                tile = image.crop((x, y, x + tile_size, y + tile_size))
                
                # Construct the filename: {unique_id}_{x}_{y}.png
                tile_filename = f"{unique_id}_{x}_{y}.png"
                tile_path = os.path.join(output_folder, tile_filename)
                
                tile.save(tile_path, format="PNG")
                
                tile_count += 1
                pbar.update(1)

    total_time = time.time() - start_time  # End timing

    # Compute performance metrics
    cpu_min, cpu_max, cpu_avg = min(cpu_usage), max(cpu_usage), sum(cpu_usage) / len(cpu_usage)
    ram_min, ram_max, ram_avg = min(ram_usage), max(ram_usage), sum(ram_usage) / len(ram_usage)

    # Output 
    print(f"\nCreated {tile_count} tiles in '{output_folder}'.")
    print(f"Total time taken: {total_time:.2f} seconds.")
    print(f"CPU Usage: Min {cpu_min:.2f}%, Max {cpu_max:.2f}%, Avg {cpu_avg:.2f}%")
    print(f"RAM Usage: Min {ram_min:.2f} MB, Max {ram_max:.2f} MB, Avg {ram_avg:.2f} MB")


# use
input_file - "data/image.tif"
output_folder = "data/tiles128"    
create_image_tiles(input_file, output_folder,tile_size=128, file=crop_pic)

Creating tiles using metadata csv file for tile coordinates


In [ ]:
def generate_tiles_from_csv(image_path, metadata_csv, output_folder):

    image = Image.open(image_path)
    df = pd.read_csv(metadata_csv)

    required_cols = {"tile_id", "x", "y", "tile_size"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

    # Read tile size once (same for all rows)
    tile_size = int(df["tile_size"].iloc[0])
    half = tile_size / 2.0

    os.makedirs(output_folder, exist_ok=True)

    print(f"Generating tiles of size {tile_size}×{tile_size} using metadata '{metadata_csv}'...")

    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Cropping Tiles"):
        tile_id = row.tile_id
        x_center = row.x
        y_center = row.y

        x0 = int(round(x_center - half))
        y0 = int(round(y_center - half))
        x1 = x0 + tile_size
        y1 = y0 + tile_size

        tile = image.crop((x0, y0, x1, y1))
        tile.save(os.path.join(output_folder, f"{tile_id}.png"))

    print(f"Done. Saved {len(df)} tiles to '{output_folder}'.")

generate_tiles_from_csv(
    image_path="data/image.tif",
    metadata_csv="metadata/metadata.csv",
    output_folder="data/tiles128"
)

In [ ]:
import os
import torch
from PIL import Image
from tqdm import tqdm
import pandas as pd  

def extract_uni_features_to_csv(
    tiles_folder,
    model,
    transform,
    device,
    batch_size=32,
    csv_path=None,
):
    # Collect all PNG tile paths
    all_files = [f for f in os.listdir(tiles_folder) if f.lower().endswith(".png")]

    # Sort numerically (tile1.png, tile2.png,)
    def tile_sort_key(name):
        base = os.path.splitext(name)[0]
        idx_str = base.replace("tile", "")
        try:
            return int(idx_str)
        except ValueError:
            return 0

    all_files = sorted(all_files, key=tile_sort_key)
    tile_paths = [os.path.join(tiles_folder, f) for f in all_files]
    tile_names = all_files

    print(f"Found {len(tile_paths)} tiles in {tiles_folder}")

    all_features = []

    model.eval()
    with torch.inference_mode():
        for i in tqdm(range(0, len(tile_paths), batch_size), desc="Extracting features"):
            batch_paths = tile_paths[i : i + batch_size]
            batch_imgs = []

            for p in batch_paths:
                img = Image.open(p).convert("RGB")
                tensor = transform(img)
                batch_imgs.append(tensor)

            batch_tensor = torch.stack(batch_imgs, dim=0).to(device)

            feats = model(batch_tensor)   # [B, D]
            feats = feats.cpu()

            all_features.append(feats)

    features = torch.cat(all_features, dim=0).numpy()  
    print("Final feature tensor shape:", features.shape)

    n_features = features.shape[1]
    col_names = [f"emb_{i+1}" for i in range(n_features)]
    df_features = pd.DataFrame(features, columns=col_names, index=tile_names)

    # Save CSV
    if csv_path is None:
        csv_path = os.path.join(tiles_folder, "uni2h_features.csv")

    df_features.to_csv(csv_path)
    print(f"✅ Embeddings saved to: {csv_path}")

    return df_features


In [ ]:
tiles_folder = "data/tiles128"
csv_path = "A:/TILES/128/uni2h_features.csv"

df = extract_uni_features_to_csv(
    tiles_folder=tiles_folder,
    model=model,
    transform=transform,
    device=device,
    batch_size=64,
    csv_path=csv_path,
)